In [1]:
# Step 1: Data Understanding & Ingestion

# Install required packages (run only once in your environment)
%pip install pandas numpy matplotlib seaborn

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# consts
file_path = "../req/MediCare_Readmission_Intelligence.csv"
cleaned_file_path = "../temp/cleaned_dataSet.csv"

In [ ]:
# Load the dataset
df = pd.read_csv(file_path)

In [3]:
# Basic inspection
print("Shape of dataset:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)

Shape of dataset: (50500, 31)

Column names: ['patient_id', 'age', 'gender', 'bmi', 'smoking_status', 'diabetes_flag', 'hypertension_flag', 'heart_disease_flag', 'chronic_conditions_count', 'previous_admissions_12m', 'length_of_stay_days', 'icu_admission_flag', 'emergency_admission_flag', 'number_of_procedures', 'blood_glucose', 'cholesterol_level', 'hemoglobin', 'creatinine', 'medications_count', 'high_risk_medication_flag', 'medication_changes_during_stay', 'followup_scheduled_flag', 'discharge_destination', 'patient_education_score', 'insurance_type', 'treatment_cost', 'legacy_record_id', 'billing_reference_number', 'care_cluster_id', 'administrative_batch_id', 'readmission_flag']

Data types:
 patient_id                          int64
age                                 int64
gender                             object
bmi                               float64
smoking_status                     object
diabetes_flag                       int64
hypertension_flag                   int64

In [4]:
# Preview first 5 rows
print("\nSample rows:")
print(df.head())


Sample rows:
   patient_id  age gender        bmi smoking_status  diabetes_flag  \
0      100000   69   Male  30.077885          Never              0   
1      100001   32   Male  22.247688        Current              1   
2      100002   89   Male  29.735513          Never              0   
3      100003   78   Male  35.185393        Current              1   
4      100004   38   Male  33.664362          Never              0   

   hypertension_flag  heart_disease_flag  chronic_conditions_count  \
0                  1                   1                         1   
1                  1                   0                         1   
2                  1                   0                         3   
3                  1                   1                         1   
4                  0                   0                         2   

   previous_admissions_12m  ...  followup_scheduled_flag  \
0                        3  ...                        1   
1                       

In [5]:
# Check target variable distribution
if 'readmission_flag' in df.columns:
    print("\nReadmission Flag Distribution:")
    print(df['readmission_flag'].value_counts())

# Summary statistics
print("\nSummary statistics:")
print(df.describe(include='all'))



Readmission Flag Distribution:
readmission_flag
0    32989
1    17511
Name: count, dtype: int64

Summary statistics:
           patient_id           age gender           bmi smoking_status  \
count    50500.000000  50500.000000  50500  50500.000000          50500   
unique            NaN           NaN      2           NaN              3   
top               NaN           NaN   Male           NaN         Former   
freq              NaN           NaN  25298           NaN          16958   
mean    124997.002455     53.378416    NaN     27.058083            NaN   
std      14438.162760     20.793392    NaN      5.868455            NaN   
min     100000.000000     18.000000    NaN     15.000000            NaN   
25%     112484.750000     35.000000    NaN     22.980879            NaN   
50%     124998.500000     53.000000    NaN     27.003718            NaN   
75%     137504.250000     71.000000    NaN     31.054301            NaN   
max     149999.000000     89.000000    NaN     50.399782 

# Key take aways from the above

Rows: 50500 rows
Columns: 31 
Target Variable: readmission_flag --> 32989 No and 17511 Yes (approx 35%)
Features: Mix of numerical (BP, sugar, cholesterol, etc), categorical (gender, insurance_type, smoking_status, discharge_destination) and administrative IDs.

Dataset is imbalanced

Several categorical features need normalization (insurance casing, discharge destinations)

Administrative IDs (legacy_record_id, billing_reference_number, care_cluster_id, administrative_batch_id) are likely non-predictive noise and should be dropped later.



In [19]:
# Step 2: Data Cleaning & Preprocessing

# 1. Remove duplicate rows
print("Rows before duplicate removal:", df.shape[0])
df = df.drop_duplicates()
print("Rows after duplicate removal:", df.shape[0])

Rows before duplicate removal: 50500
Rows after duplicate removal: 50000


In [ ]:
# 2. Standardize categorical values 

cat_cols = [
    "gender", "smoking_status", "insurance_type", "discharge_destination"
]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()


In [27]:
# 3. Data type corrections
#['patient_id', 'age', 'gender', 'bmi', 'smoking_status', 'diabetes_flag', 'hypertension_flag', 'heart_disease_flag', 'chronic_conditions_count', 'previous_admissions_12m', 'length_of_stay_days', 'icu_admission_flag', 'emergency_admission_flag', 'number_of_procedures', 'blood_glucose', 'cholesterol_level', 'hemoglobin', 'creatinine', 'medications_count', 'high_risk_medication_flag', 'medication_changes_during_stay', 'followup_scheduled_flag', 'discharge_destination', 'patient_education_score', 'insurance_type', 'treatment_cost', 'legacy_record_id', 'billing_reference_number', 'care_cluster_id', 'administrative_batch_id', 'readmission_flag']


numeric_cols = [
    "age", 
    "bmi", 
    "chronic_conditions_count", 
    "previous_admissions_12m", 
    "length_of_stay_days", 
    "number_of_procedures", 
    "blood_glucose", 
    "cholesterol_level", 
    "hemoglobin", 
    "creatinine", 
    "medications_count",
    "patient_education_score",
    "treatment_cost"
]

binary_cols = [
    "diabetes_flag",
    "hypertension_flag",
    "heart_disease_flag",
    "icu_admission_flag",
    "emergency_admission_flag",
    "high_risk_medication_flag",
    "followup_scheduled_flag",
    "readmission_flag"
]

# Convert numeric columns to appropriate data types
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Convert binary columns to numeric
df[binary_cols] = df[binary_cols].apply(pd.to_numeric, errors='coerce')

In [ ]:
# 4. Missing data handling

# % missing check
missing_pct = df.isnull().mean() * 100
print("\nMissing data percentage per column:\n", missing_pct[missing_pct > 0])

# Numerical columns: Impute missing values with median
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col].fillna(median_value, inplace=True)
        print(f"Imputed missing values in {col} with median: {median_value}")

# Categorical columns: Impute missing values with mode
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].mode()[0]
        df[col].fillna(mode_value, inplace=True)
        print(f"Imputed missing values in {col} with mode: {mode_value}")

# Verify no missing values remain
print("\nMissing data after imputation:\n", df.isnull().sum())



Missing data percentage per column:
 Series([], dtype: float64)

Missing data after imputation:
 patient_id                        0
age                               0
gender                            0
bmi                               0
smoking_status                    0
diabetes_flag                     0
hypertension_flag                 0
heart_disease_flag                0
chronic_conditions_count          0
previous_admissions_12m           0
length_of_stay_days               0
icu_admission_flag                0
emergency_admission_flag          0
number_of_procedures              0
blood_glucose                     0
cholesterol_level                 0
hemoglobin                        0
creatinine                        0
medications_count                 0
high_risk_medication_flag         0
medication_changes_during_stay    0
followup_scheduled_flag           0
discharge_destination             0
patient_education_score           0
insurance_type                    0
tr

In [33]:
# 5. Outlier detection and handling - Winsorization

def cap_outliers(series, lower = 0.01, upper = 0.99):
    lower_bound = series.quantile(lower)
    upper_bound = series.quantile(upper)
    return series.clip(lower=lower_bound, upper=upper_bound)

for col in numeric_cols:
    df[col] = cap_outliers(df[col])

In [35]:
# 6. Establishing rules for data validation

#Idea is to remove impossible values, 
df = df[(df['age'] >= 0)]  # Age cannot be negative
df = df[(df['bmi'] >= 0) & (df['bmi'] <= 10)] # BMI should be in a reasonable range (0-10 for this dataset)
df = df[df['length_of_stay_days'] >= 0]  # Length of stay cannot be negative
df = df[df['blood_glucose'] >= 0]  # Blood glucose cannot be negative
df = df[df['cholesterol_level'] >= 0]  # Cholesterol level cannot be negative

# Capping extreme realistic values
df["age"] = np.clip(df["age"], 0, 120)  # Age should be between 0 and 120
df["bmi"] = np.clip(df["bmi"], 10, 60)  # BMI should be between 10 and 60

In [ ]:
# Final Sanity Check
print("\nFinal dataset shape after cleaning:", df.shape)


Final dataset shape after cleaning: (0, 31)


In [ ]:
# Save the cleaned dataset for further analysis
df.to_csv(cleaned_file_path, index=False)